# Phase 1

In [ ]:
# ============================================================
# STRICT REDUNDANCY / DUPLICATION ANALYSIS

# Expected structure:
#   4 LLM folders × 5 VLM folders × 29 CSV files = 580 test-case CSV files
#
# STRICT RULE:
#   Confirmed duplicate = normalized FULL ROW is exactly the same.
#   Same title alone is NOT duplicate.
#   Same title + different steps is NOT duplicate.
#   Same title + different expected result is NOT duplicate.
#   Similar semantic pair is saved only as "candidate for manual review",
#   not counted as confirmed duplicate.
#
# Outputs saved inside:
#   ROOT_INPUT_DIR/redundancy_analysis_strict/
#   ROOT_INPUT_DIR/redundancy_analysis_strict.zip
# ============================================================

# =========================
# 1. Install dependencies
# =========================

!pip -q install \
    "huggingface_hub==0.25.2" \
    "transformers==4.44.2" \
    "sentence-transformers==3.0.1" \
    "scikit-learn" \
    "openpyxl" \
    "matplotlib" \
    "pandas" \
    "numpy"

# =========================
# 2. Mount Drive and imports
# =========================

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

# =========================
# 3. Main folder path
# =========================
# Update only this path if your Drive folder name is different.

ROOT_INPUT_DIR = "/content/drive/MyDrive/SEMANTIC_COVERAGE_READY"

if not os.path.exists(ROOT_INPUT_DIR):
    candidates = [
        os.path.join("/content/drive/MyDrive", d)
        for d in os.listdir("/content/drive/MyDrive")
        if d.lower().startswith("semantic_coverag")
    ]

    print("Candidate SEMANTIC_COVERAGE folders found:")
    for c in candidates:
        print(" ", c)

    if len(candidates) == 1:
        ROOT_INPUT_DIR = candidates[0]
        print("\nUsing detected folder:", ROOT_INPUT_DIR)
    else:
        raise RuntimeError(
            "Could not automatically find the SEMANTIC_COVERAGE folder. "
            "Please set ROOT_INPUT_DIR manually."
        )

OUT_DIR = os.path.join(ROOT_INPUT_DIR, "redundancy_analysis_strict")
DEDUP_DIR = os.path.join(OUT_DIR, "exact_deduplicated_testcases")
FIG_DIR = os.path.join(OUT_DIR, "figures")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DEDUP_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Input root:", ROOT_INPUT_DIR)
print("Output root:", OUT_DIR)

# =========================
# 4. Settings
# =========================

EXPECTED_LLM_COUNT = 4
EXPECTED_VLM_COUNT_PER_LLM = 5
EXPECTED_REPORTS_PER_VLM = 29
EXPECTED_TOTAL_CSV = EXPECTED_LLM_COUNT * EXPECTED_VLM_COUNT_PER_LLM * EXPECTED_REPORTS_PER_VLM

# Semantic similarity is only for manual-review candidates.
# It is NOT used as confirmed duplicate evidence.
CANDIDATE_THRESHOLDS = [0.90, 0.95]
MAIN_CANDIDATE_THRESHOLD = 0.90

EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
CACHE_DIR = "/content/bge_large_cache"

def threshold_key(t):
    return f"{t:.2f}".replace(".", "_")

MAIN_KEY = threshold_key(MAIN_CANDIDATE_THRESHOLD)

# =========================
# 5. CSV reading and normalization helpers
# =========================

def robust_read_csv(path):
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
        if len(df.columns) > 1:
            return df
    except Exception:
        pass

    try:
        df = pd.read_csv(path, sep=";", encoding="utf-8-sig", engine="python")
        if len(df.columns) > 1:
            return df
    except Exception:
        pass

    return pd.read_csv(path, encoding="utf-8-sig", engine="python")


def clean_text_basic(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_full_row_for_exact(text):
    """
    Strict exact duplicate normalization.
    This removes only IDs, numbering, punctuation, and spacing.
    It does NOT remove actual test meaning.
    Therefore, different steps/expected results remain different.
    """
    text = "" if text is None or pd.isna(text) else str(text)
    text = text.lower()

    # Remove generated IDs and numbering only.
    text = re.sub(r"\btc[-_a-z0-9]*[-_]\d+\b", " ", text)
    text = re.sub(r"\btest case id\s*:\s*\S+", " ", text)
    text = re.sub(r"\b\d+\.\s*", " ", text)

    # Normalize punctuation and spacing.
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_col(row, candidates):
    for c in candidates:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c]).strip()
    return ""


def build_full_row_text(row):
    """
    Full test-case text used for exact duplicate detection.
    Uses all meaningful columns, not title only.
    """
    tcid = get_col(row, ["Test Case ID", "test_case_id", "testcase_id", "ID", "id"])
    title = get_col(row, ["Title", "title", "Test Case Title", "test_title", "Name", "name"])
    module = get_col(row, ["Module", "module", "Feature", "feature", "Component", "component"])
    source_pages = get_col(row, ["Source Pages", "source_pages", "Page", "Pages"])
    preconditions = get_col(row, ["Preconditions", "Precondition", "preconditions", "precondition"])
    test_data = get_col(row, ["Test Data", "test_data", "Data", "data"])
    steps = get_col(row, ["Test Steps", "Steps", "steps", "Procedure", "procedure"])
    expected = get_col(row, ["Expected Result", "Expected Results", "expected_result", "expected_results", "Oracle", "oracle"])

    known_text = (
        f"Test Case ID: {tcid}. "
        f"Title: {title}. "
        f"Module: {module}. "
        f"Source Pages: {source_pages}. "
        f"Preconditions: {preconditions}. "
        f"Test Data: {test_data}. "
        f"Test Steps: {steps}. "
        f"Expected Result: {expected}."
    )

    # If standard fields are absent, use every cell in the row.
    meaningful_known = clean_text_basic(
        f"{title} {module} {preconditions} {test_data} {steps} {expected}"
    )

    if meaningful_known:
        return clean_text_basic(known_text)

    all_cells = " ".join([
        str(x) for x in row.values
        if pd.notna(x) and str(x).strip()
    ])

    return clean_text_basic(all_cells)


def build_candidate_review_text(row):
    """
    Used only for semantic candidate detection.
    Candidate pairs are not counted as confirmed duplicates.
    """
    title = get_col(row, ["Title", "title", "Test Case Title", "test_title", "Name", "name"])
    module = get_col(row, ["Module", "module", "Feature", "feature", "Component", "component"])
    preconditions = get_col(row, ["Preconditions", "Precondition", "preconditions", "precondition"])
    test_data = get_col(row, ["Test Data", "test_data", "Data", "data"])
    steps = get_col(row, ["Test Steps", "Steps", "steps", "Procedure", "procedure"])
    expected = get_col(row, ["Expected Result", "Expected Results", "expected_result", "expected_results", "Oracle", "oracle"])

    text = (
        f"Title: {title}. "
        f"Module: {module}. "
        f"Preconditions: {preconditions}. "
        f"Test Data: {test_data}. "
        f"Test Steps: {steps}. "
        f"Expected Result: {expected}."
    )

    if clean_text_basic(text):
        return clean_text_basic(text)

    return build_full_row_text(row)


def row_hash_from_text(text):
    import hashlib
    return hashlib.md5(normalize_full_row_for_exact(text).encode("utf-8")).hexdigest()


def report_id_from_filename(path):
    name = os.path.basename(path)
    name = name.replace("_testcases.csv", "")
    name = name.replace(".csv", "")
    return name


# =========================
# 6. Discover files: LLM / VLM / Report CSV
# =========================

def discover_testcase_files(root_dir):
    rows = []

    skip_folders = {
        "redundancy_analysis",
        "redundancy_analysis_field_aware",
        "redundancy_analysis_strict",
        "__MACOSX",
        ".ipynb_checkpoints"
    }

    llm_folders = [
        d for d in sorted(os.listdir(root_dir))
        if os.path.isdir(os.path.join(root_dir, d))
        and d not in skip_folders
        and not d.startswith(".")
    ]

    for llm_name in llm_folders:
        llm_path = os.path.join(root_dir, llm_name)

        vlm_folders = [
            d for d in sorted(os.listdir(llm_path))
            if os.path.isdir(os.path.join(llm_path, d))
            and d not in skip_folders
            and "redundancy" not in d.lower()
            and not d.startswith(".")
        ]

        for vlm_name in vlm_folders:
            vlm_path = os.path.join(llm_path, vlm_name)

            for current_root, dirs, files in os.walk(vlm_path):
                dirs[:] = [
                    d for d in dirs
                    if "semantic_coverage_output" not in d.lower()
                    and "deduplicated" not in d.lower()
                    and "redundancy" not in d.lower()
                    and "__macosx" not in d.lower()
                    and ".ipynb_checkpoints" not in d.lower()
                ]

                for f in files:
                    lower = f.lower()

                    if not lower.endswith(".csv"):
                        continue

                    if "_testcases" not in lower:
                        continue

                    full_path = os.path.join(current_root, f)

                    rows.append({
                        "llm": llm_name,
                        "vlm": vlm_name,
                        "report_id": report_id_from_filename(full_path),
                        "csv_file": f,
                        "csv_path": full_path
                    })

    return pd.DataFrame(rows)


files_df = discover_testcase_files(ROOT_INPUT_DIR)

print("\nDiscovered test-case CSV files:", len(files_df))
print("Expected test-case CSV files:", EXPECTED_TOTAL_CSV)

if len(files_df) == 0:
    raise RuntimeError("No *_testcases.csv files found. Please check ROOT_INPUT_DIR.")

files_df.to_csv(os.path.join(OUT_DIR, "discovered_testcase_files.csv"), index=False)

# Structure validation
structure_llm_df = (
    files_df.groupby("llm")
    .agg(
        vlm_count=("vlm", "nunique"),
        csv_count=("csv_path", "count"),
        report_count=("report_id", "nunique")
    )
    .reset_index()
)

structure_vlm_df = (
    files_df.groupby(["llm", "vlm"])
    .agg(
        csv_count=("csv_path", "count"),
        report_count=("report_id", "nunique")
    )
    .reset_index()
)

structure_llm_df.to_csv(os.path.join(OUT_DIR, "structure_check_by_llm.csv"), index=False)
structure_vlm_df.to_csv(os.path.join(OUT_DIR, "structure_check_by_llm_vlm.csv"), index=False)

print("\nStructure by LLM:")
display(structure_llm_df)

print("\nStructure by LLM-VLM:")
display(structure_vlm_df)

if len(files_df) != EXPECTED_TOTAL_CSV:
    print("\nWARNING:")
    print("The number of discovered CSV files is not 580.")
    print("This does not stop the analysis, but check structure_check_by_llm_vlm.csv.")

# =========================
# 7. Load BGE only for candidate-review pairs
# =========================

print("\nLoading BGE-large for semantic candidate review only:", EMBEDDING_MODEL_NAME)

bge = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    cache_folder=CACHE_DIR,
    model_kwargs={"use_safetensors": False}
)

print("BGE loaded.")

# =========================
# 8. Analyze one test-case CSV
# =========================

def analyze_one_csv(llm_name, vlm_name, report_id, csv_path):
    raw_df = robust_read_csv(csv_path).copy().reset_index(drop=True)

    if len(raw_df) == 0:
        return None, pd.DataFrame(), pd.DataFrame()

    df = raw_df.copy()
    df["llm"] = llm_name
    df["vlm"] = vlm_name
    df["report_id"] = report_id
    df["source_csv"] = csv_path

    df["full_row_text"] = df.apply(build_full_row_text, axis=1)
    df["full_row_norm"] = df["full_row_text"].apply(normalize_full_row_for_exact)
    df["full_row_hash"] = df["full_row_text"].apply(row_hash_from_text)
    df["candidate_review_text"] = df.apply(build_candidate_review_text, axis=1)

    n = len(df)

    # Confirmed exact duplicates only
    exact_unique_cases = df["full_row_hash"].nunique()
    exact_duplicate_count = n - exact_unique_cases
    exact_duplicate_rate = exact_duplicate_count / n if n else 0.0
    exact_unique_ratio = exact_unique_cases / n if n else 0.0

    duplicate_groups = (
        df.groupby("full_row_hash")
        .filter(lambda x: len(x) > 1)
        .sort_values(["full_row_hash"])
    )

    confirmed_duplicate_rows = []

    if len(duplicate_groups) > 0:
        for h, group in duplicate_groups.groupby("full_row_hash"):
            indices = group.index.tolist()
            representative_idx = indices[0]

            for idx in indices[1:]:
                confirmed_duplicate_rows.append({
                    "llm": llm_name,
                    "vlm": vlm_name,
                    "report_id": report_id,
                    "duplicate_type": "confirmed_exact_duplicate",
                    "row_i_index": representative_idx,
                    "row_j_index": idx,
                    "row_i_text": df.loc[representative_idx, "full_row_text"],
                    "row_j_text": df.loc[idx, "full_row_text"],
                    "normalized_text": df.loc[idx, "full_row_norm"],
                    "source_csv": csv_path
                })

    confirmed_duplicate_pairs_df = pd.DataFrame(confirmed_duplicate_rows)

    # Exact deduplicated file: keep first occurrence of identical full-row hash
    exact_dedup_df = df.drop_duplicates(subset=["full_row_hash"], keep="first").copy().reset_index(drop=True)

    out_subdir = os.path.join(DEDUP_DIR, llm_name, vlm_name, report_id)
    os.makedirs(out_subdir, exist_ok=True)

    exact_dedup_path = os.path.join(out_subdir, f"{report_id}_exact_deduplicated_testcases.csv")
    exact_dedup_df.to_csv(exact_dedup_path, index=False)

    # Semantic candidate pairs for manual review only
    candidate_rows = []

    if n > 1:
        embeddings = bge.encode(
            df["candidate_review_text"].tolist(),
            batch_size=16,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        sim = cosine_similarity(embeddings)
        np.fill_diagonal(sim, -1)

        nearest = sim.max(axis=1)
        avg_nearest_similarity = float(np.mean(nearest))
        max_nearest_similarity = float(np.max(nearest))

        total_pairs = n * (n - 1) / 2

        for tau in CANDIDATE_THRESHOLDS:
            candidate_count = 0

            for i in range(n):
                for j in range(i + 1, n):
                    score = float(sim[i, j])

                    if score >= tau:
                        candidate_count += 1

                        if tau == MAIN_CANDIDATE_THRESHOLD:
                            candidate_rows.append({
                                "llm": llm_name,
                                "vlm": vlm_name,
                                "report_id": report_id,
                                "threshold": tau,
                                "pair_type": "semantic_candidate_for_manual_review_only",
                                "row_i_index": i,
                                "row_j_index": j,
                                "similarity": score,
                                "row_i_text": df.iloc[i]["full_row_text"],
                                "row_j_text": df.iloc[j]["full_row_text"],
                                "same_exact_hash": bool(df.iloc[i]["full_row_hash"] == df.iloc[j]["full_row_hash"]),
                                "source_csv": csv_path
                            })

            candidate_rate = candidate_count / total_pairs if total_pairs else 0.0

            # Save in summary dynamically
            if tau == MAIN_CANDIDATE_THRESHOLD:
                candidate_pair_count_090 = candidate_count
                candidate_pair_rate_090 = candidate_rate
            elif tau == 0.95:
                candidate_pair_count_095 = candidate_count
                candidate_pair_rate_095 = candidate_rate

    else:
        avg_nearest_similarity = np.nan
        max_nearest_similarity = np.nan
        candidate_pair_count_090 = 0
        candidate_pair_rate_090 = 0.0
        candidate_pair_count_095 = 0
        candidate_pair_rate_095 = 0.0

    semantic_candidate_df = pd.DataFrame(candidate_rows)

    summary = {
        "llm": llm_name,
        "vlm": vlm_name,
        "report_id": report_id,
        "csv_path": csv_path,
        "num_generated_cases": n,

        # Main confirmed duplicate metrics
        "confirmed_exact_unique_cases": exact_unique_cases,
        "confirmed_exact_duplicate_count": exact_duplicate_count,
        "confirmed_exact_duplicate_rate": exact_duplicate_rate,
        "confirmed_exact_unique_ratio": exact_unique_ratio,

        # Candidate metrics only, not confirmed duplicates
        "semantic_candidate_pair_count_at_0_90": int(candidate_pair_count_090),
        "semantic_candidate_pair_rate_at_0_90": float(candidate_pair_rate_090),
        "semantic_candidate_pair_count_at_0_95": int(candidate_pair_count_095),
        "semantic_candidate_pair_rate_at_0_95": float(candidate_pair_rate_095),
        "avg_nearest_neighbor_similarity_for_review": avg_nearest_similarity,
        "max_nearest_neighbor_similarity_for_review": max_nearest_similarity,

        # Dedup only exact duplicates
        "exact_deduplicated_cases": len(exact_dedup_df),
        "case_reduction_after_exact_dedup": 1 - (len(exact_dedup_df) / n) if n else 0.0,
        "exact_deduplicated_csv_path": exact_dedup_path
    }

    return summary, confirmed_duplicate_pairs_df, semantic_candidate_df


# =========================
# 9. Run analysis over all 580 CSV files
# =========================

summary_rows = []
confirmed_duplicate_pair_tables = []
semantic_candidate_tables = []
error_rows = []

for _, row in files_df.iterrows():
    llm_name = row["llm"]
    vlm_name = row["vlm"]
    report_id = row["report_id"]
    csv_path = row["csv_path"]

    print(f"Analyzing: LLM={llm_name} | VLM={vlm_name} | report={report_id}")

    try:
        summary, confirmed_pairs_df, candidate_df = analyze_one_csv(
            llm_name=llm_name,
            vlm_name=vlm_name,
            report_id=report_id,
            csv_path=csv_path
        )

        if summary is not None:
            summary_rows.append(summary)

        if confirmed_pairs_df is not None and len(confirmed_pairs_df) > 0:
            confirmed_duplicate_pair_tables.append(confirmed_pairs_df)

        if candidate_df is not None and len(candidate_df) > 0:
            semantic_candidate_tables.append(candidate_df)

    except Exception as e:
        print("FAILED:", csv_path, type(e).__name__, str(e)[:500])

        error_rows.append({
            "llm": llm_name,
            "vlm": vlm_name,
            "report_id": report_id,
            "csv_path": csv_path,
            "error_type": type(e).__name__,
            "error_message": str(e)
        })

summary_df = pd.DataFrame(summary_rows)

if len(summary_df) == 0:
    raise RuntimeError("No files were successfully analyzed.")

if confirmed_duplicate_pair_tables:
    confirmed_pairs_df = pd.concat(confirmed_duplicate_pair_tables, ignore_index=True)
else:
    confirmed_pairs_df = pd.DataFrame(columns=[
        "llm", "vlm", "report_id", "duplicate_type",
        "row_i_index", "row_j_index", "row_i_text", "row_j_text",
        "normalized_text", "source_csv"
    ])

if semantic_candidate_tables:
    semantic_candidates_df = pd.concat(semantic_candidate_tables, ignore_index=True)
else:
    semantic_candidates_df = pd.DataFrame(columns=[
        "llm", "vlm", "report_id", "threshold", "pair_type",
        "row_i_index", "row_j_index", "similarity",
        "row_i_text", "row_j_text", "same_exact_hash", "source_csv"
    ])

errors_df = pd.DataFrame(error_rows)

# =========================
# 10. Summary matrices
# =========================

# LLM-level main matrix
llm_summary_df = (
    summary_df
    .groupby("llm")
    .agg(
        vlm_count=("vlm", "nunique"),
        csv_files=("csv_path", "count"),
        reports=("report_id", "nunique"),
        total_generated_cases=("num_generated_cases", "sum"),
        confirmed_exact_duplicate_cases=("confirmed_exact_duplicate_count", "sum"),
        total_exact_deduplicated_cases=("exact_deduplicated_cases", "sum"),
        mean_confirmed_exact_duplicate_rate=("confirmed_exact_duplicate_rate", "mean"),
        mean_confirmed_exact_unique_ratio=("confirmed_exact_unique_ratio", "mean"),
        mean_semantic_candidate_pair_rate_at_0_90=("semantic_candidate_pair_rate_at_0_90", "mean"),
        mean_semantic_candidate_pair_rate_at_0_95=("semantic_candidate_pair_rate_at_0_95", "mean"),
        mean_avg_nearest_neighbor_similarity_for_review=("avg_nearest_neighbor_similarity_for_review", "mean"),
    )
    .reset_index()
)

llm_summary_df["overall_exact_case_reduction_rate"] = (
    1 - llm_summary_df["total_exact_deduplicated_cases"] / llm_summary_df["total_generated_cases"]
)

# LLM × VLM matrix
llm_vlm_summary_df = (
    summary_df
    .groupby(["llm", "vlm"])
    .agg(
        csv_files=("csv_path", "count"),
        reports=("report_id", "nunique"),
        total_generated_cases=("num_generated_cases", "sum"),
        confirmed_exact_duplicate_cases=("confirmed_exact_duplicate_count", "sum"),
        total_exact_deduplicated_cases=("exact_deduplicated_cases", "sum"),
        mean_confirmed_exact_duplicate_rate=("confirmed_exact_duplicate_rate", "mean"),
        mean_confirmed_exact_unique_ratio=("confirmed_exact_unique_ratio", "mean"),
        mean_semantic_candidate_pair_rate_at_0_90=("semantic_candidate_pair_rate_at_0_90", "mean"),
        mean_semantic_candidate_pair_rate_at_0_95=("semantic_candidate_pair_rate_at_0_95", "mean"),
    )
    .reset_index()
)

llm_vlm_summary_df["overall_exact_case_reduction_rate"] = (
    1 - llm_vlm_summary_df["total_exact_deduplicated_cases"] / llm_vlm_summary_df["total_generated_cases"]
)

# Main paper matrix: short and safe
paper_matrix_df = llm_summary_df[[
    "llm",
    "vlm_count",
    "csv_files",
    "total_generated_cases",
    "confirmed_exact_duplicate_cases",
    "mean_confirmed_exact_duplicate_rate",
    "mean_confirmed_exact_unique_ratio",
    "overall_exact_case_reduction_rate",
    "mean_semantic_candidate_pair_rate_at_0_90",
    "mean_semantic_candidate_pair_rate_at_0_95",
]].copy()

paper_matrix_df = paper_matrix_df.rename(columns={
    "llm": "LLM",
    "vlm_count": "VLM sources",
    "csv_files": "CSV files analyzed",
    "total_generated_cases": "Generated test cases",
    "confirmed_exact_duplicate_cases": "Confirmed exact duplicate cases",
    "mean_confirmed_exact_duplicate_rate": "Mean exact duplicate rate",
    "mean_confirmed_exact_unique_ratio": "Mean exact unique-case ratio",
    "overall_exact_case_reduction_rate": "Overall exact deduplication reduction",
    "mean_semantic_candidate_pair_rate_at_0_90": "Semantic candidate pair rate@0.90",
    "mean_semantic_candidate_pair_rate_at_0_95": "Semantic candidate pair rate@0.95",
})

# =========================
# 11. Save CSV outputs
# =========================

paths = {
    "discovered_files": os.path.join(OUT_DIR, "discovered_testcase_files.csv"),
    "structure_by_llm": os.path.join(OUT_DIR, "structure_check_by_llm.csv"),
    "structure_by_llm_vlm": os.path.join(OUT_DIR, "structure_check_by_llm_vlm.csv"),
    "per_csv_summary": os.path.join(OUT_DIR, "strict_redundancy_per_csv.csv"),
    "llm_summary": os.path.join(OUT_DIR, "strict_redundancy_llm_summary.csv"),
    "llm_vlm_summary": os.path.join(OUT_DIR, "strict_redundancy_llm_vlm_summary.csv"),
    "paper_matrix": os.path.join(OUT_DIR, "strict_redundancy_paper_matrix.csv"),
    "confirmed_pairs": os.path.join(OUT_DIR, "confirmed_exact_duplicate_pairs.csv"),
    "semantic_candidates": os.path.join(OUT_DIR, "semantic_similarity_candidates_for_manual_review.csv"),
    "errors": os.path.join(OUT_DIR, "strict_redundancy_errors.csv"),
}

files_df.to_csv(paths["discovered_files"], index=False)
structure_llm_df.to_csv(paths["structure_by_llm"], index=False)
structure_vlm_df.to_csv(paths["structure_by_llm_vlm"], index=False)
summary_df.to_csv(paths["per_csv_summary"], index=False)
llm_summary_df.to_csv(paths["llm_summary"], index=False)
llm_vlm_summary_df.to_csv(paths["llm_vlm_summary"], index=False)
paper_matrix_df.to_csv(paths["paper_matrix"], index=False)
confirmed_pairs_df.to_csv(paths["confirmed_pairs"], index=False)
semantic_candidates_df.to_csv(paths["semantic_candidates"], index=False)
errors_df.to_csv(paths["errors"], index=False)

# =========================
# 12. Save Excel workbook
# =========================

xlsx_path = os.path.join(OUT_DIR, "c10_strict_redundancy_results.xlsx")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    paper_matrix_df.to_excel(writer, sheet_name="paper_matrix", index=False)
    llm_summary_df.to_excel(writer, sheet_name="llm_summary", index=False)
    llm_vlm_summary_df.to_excel(writer, sheet_name="llm_vlm_summary", index=False)
    summary_df.to_excel(writer, sheet_name="per_csv_summary", index=False)
    confirmed_pairs_df.to_excel(writer, sheet_name="confirmed_exact_pairs", index=False)
    semantic_candidates_df.head(5000).to_excel(writer, sheet_name="semantic_candidates_sample", index=False)
    structure_llm_df.to_excel(writer, sheet_name="structure_by_llm", index=False)
    structure_vlm_df.to_excel(writer, sheet_name="structure_by_llm_vlm", index=False)
    errors_df.to_excel(writer, sheet_name="errors", index=False)

print("\nSaved Excel:", xlsx_path)

# =========================
# 13. Figures
# =========================

def save_bar(df, x_col, y_col, title, ylabel, filename, ylim=None):
    plt.figure(figsize=(8, 5))
    plt.bar(df[x_col], df[y_col])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=25, ha="right")

    if ylim is not None:
        plt.ylim(*ylim)

    for i, v in enumerate(df[y_col].tolist()):
        if pd.notna(v):
            plt.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, filename)
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)


save_bar(
    llm_summary_df,
    "llm",
    "mean_confirmed_exact_duplicate_rate",
    "Confirmed exact duplicate rate by LLM",
    "Exact duplicate rate",
    "confirmed_exact_duplicate_rate_by_llm.png",
    ylim=(0, 1.05)
)

save_bar(
    llm_summary_df,
    "llm",
    "mean_confirmed_exact_unique_ratio",
    "Confirmed exact unique-case ratio by LLM",
    "Exact unique-case ratio",
    "confirmed_exact_unique_ratio_by_llm.png",
    ylim=(0, 1.05)
)

save_bar(
    llm_summary_df,
    "llm",
    "mean_semantic_candidate_pair_rate_at_0_90",
    "Semantic similarity candidate rate by LLM at 0.90",
    "Candidate pair rate",
    "semantic_candidate_pair_rate_090_by_llm.png",
    ylim=(0, 1.05)
)

# =========================
# 14. Paper-ready text
# =========================

paper_text = """
C10 Redundancy and Duplicate Analysis

We analyzed redundancy over the generated design-level test-case specifications without regenerating any outputs. The analysis follows the actual generation structure: four test-case generation LLMs, five VLM-derived UML-description sources per LLM, and 29 report-level test-case CSV files per VLM source. Therefore, the expected input size is 4 × 5 × 29 = 580 test-case CSV files. Only files ending in _testcases.csv were included; description files and semantic-coverage output folders were ignored.

We define a confirmed duplicate conservatively as a repeated test-case row after normalizing only identifiers, numbering, punctuation, and whitespace. The full test-case content is used, including title, module, source page, preconditions, test data, test steps, and expected result when available. Therefore, two test cases with the same title but different steps, inputs, or expected outcomes are not counted as duplicates.

In addition to confirmed exact duplicates, we compute semantic similarity candidates for manual inspection. These candidates are produced using BGE-large embeddings over the full test-case representation and cosine similarity thresholds of 0.90 and 0.95. These semantic pairs are not counted as confirmed duplicates, because similar wording can still represent distinct positive, negative, or branch-specific scenarios. The 0.90 threshold is used only as a conservative high-similarity filter for candidate review, not as a universal software-testing threshold.

The reported matrix therefore separates confirmed exact duplicate metrics from semantic similarity candidate metrics. The main redundancy claim is based on confirmed exact duplicate rate and exact unique-case ratio, while semantic candidate rates are reported only as auxiliary evidence for manual review.
""".strip()

paper_text_path = os.path.join(OUT_DIR, "paper_ready_c10_strict_redundancy_text.txt")

with open(paper_text_path, "w", encoding="utf-8") as f:
    f.write(paper_text)

print("Saved paper-ready text:", paper_text_path)

# =========================
# 15. ZIP outputs
# =========================

FINAL_ZIP = os.path.join(ROOT_INPUT_DIR, "redundancy_analysis_strict.zip")

if os.path.exists(FINAL_ZIP):
    os.remove(FINAL_ZIP)

shutil.make_archive(
    FINAL_ZIP.replace(".zip", ""),
    "zip",
    OUT_DIR
)

print("\nDONE.")
print("Discovered CSV files:", len(files_df))
print("Expected CSV files:", EXPECTED_TOTAL_CSV)
print("Main output folder:", OUT_DIR)
print("Final ZIP saved in Drive:", FINAL_ZIP)

print("\nMain files:")
print(paths["paper_matrix"])
print(paths["llm_summary"])
print(paths["llm_vlm_summary"])
print(paths["confirmed_pairs"])
print(paths["semantic_candidates"])
print(xlsx_path)
print(paper_text_path)
print(FINAL_ZIP)

display(paper_matrix_df)